In [1]:
# -*- coding: utf-8 -*-
"""
============================================================
TERZINA FPA — CAMERA MAPS + BRIGHTEST-CHANNEL FINGER PLOTS
============================================================

One pass over the .bin produces both:

    plots/
        Camera_HIT_Occupancy.png      camera maps, focal-plane coordinates
        Camera_HG_Mean.png
        Camera_LG_Mean.png
        bright_fingerplots_HG.png     grid of the channels that resolve the
                                      most p.e. peaks
        bright_DAQ<d>_ASIC<a>_CH<c>_HG.png    full-size, top few

    channel_summary_<run>.csv
    complete_mapping_<nx>x<ny>.csv

Set BIN_FILE and MAPPING_DIR below, then run:

    python terzina_analysis.py
    python terzina_analysis.py path/to/run.bin

No folder juggling, no argparse, no os.chdir — absolute paths throughout,
so it works the same from a terminal, Spyder or Jupyter.

============================================================
"""

import os
import csv
import sys
import time

from array import array

import numpy as np
import pandas as pd

import matplotlib

# Only force the file-only backend outside IPython, so Spyder's Plots pane
# and %matplotlib inline still work when you want them.
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib import patches

from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from scipy.optimize import curve_fit


# ============================================================
# CONFIGURATION
# ============================================================

BIN_FILE = r"C:\Users\muhda\Desktop\Turin_Mission\Data\data_07_03-11_28.bin"

MAPPING_DIR = r"C:\Users\muhda\Desktop\Software\GitHub\Terzina_data_ana\mapping_ana"

MAPPING_FILES = [
    "finalmappingsTERZINAGlobalIDmapping(23_03_2026)-QUADRANT0.csv",
    "finalmappingsTERZINAGlobalIDmapping(23_03_2026)-QUADRANT1.csv",
    "finalmappingsTERZINAGlobalIDmapping(23_03_2026)-QUADRANT2.csv",
    "finalmappingsTERZINAGlobalIDmapping(23_03_2026)-QUADRANT3.csv",
]
CONF_FILE = "TerzinaFPA.conf"

MAX_PACKETS = None      # None = all, or e.g. 50000 for a quick look

# --- Finger-plot / peak finding --------------------------------------------
NBINS = 16384           # 14-bit ADC
REBIN_FACTOR = 2        # 2 -> 8192 bins of 2 ADC each
SMOOTH_SIGMA = 2.0      # Gaussian smoothing for peak finding
PEAK_PROMINENCE = 0.005 # as a fraction of the tallest peak
PEAK_DISTANCE = 10      # minimum peak separation, in rebinned bins
GAUSS_FIT_WINDOW = 6    # half-window for the per-peak Gaussian fit

BRIGHT_GAIN = "HG"      # gain branch for the brightest-channel search
BRIGHT_N_TOP = 32       # how many channels in the grid
BRIGHT_MIN_PEAKS = 3    # ignore channels resolving fewer peaks than this
BRIGHT_N_INDIVIDUAL = 3 # how many full-size single-channel plots to also save
BRIGHT_NCOLS = 4        # columns in the grid

# --- Known-bad ASIC ---------------------------------------------------------
# Luigi's kit has a dead ASIC that fires on every event and would always be
# picked as "brightest". Bench name DAQ 2 ASIC D = DAQ 3 ASIC D in the binary.
#   Turin data  -> True   (present and saturating: D3_D all channels at the
#                          full event count)
#   NI data     -> False  (DAQ3 not read out, nothing to exclude)
#   LNGS/Geneva -> False  (different hardware, no fault)
EXCLUDE_BAD_ASIC = True
BAD_ASICS = [(3, "D")]          # (DAQ as numbered in the binary, ASIC letter)

# ---------------------------------------------------------------------------

DAQ_IDS = {0x00AAAA00: 1, 0x00BBBB00: 2, 0x00CCCC00: 3, 0x00DDDD00: 4}
CONC_HEADER = 0xCA00FE11DD651983
VALID_PACK_ID = 0xA77A

DAQS = [1, 2, 3, 4]
ASICS = ["A", "B", "C", "D", "E"]
PREFIXES = [f"D{d}_{a}" for d in DAQS for a in ASICS]


# ============================================================
# DATA PROCESSING
# ============================================================

class DataProcessing:
    """
    Parses the .bin and holds the raw 32-bit readout words per ASIC.

    Words are kept in array('I') rather than Python lists — 4 bytes each
    instead of ~36 — and HG/LG/HIT are decoded on demand with numpy. On a
    125k-packet file that is a few hundred MB instead of several GB.

    Word layout: HG = bits 0-13, LG = bits 14-27, HIT = bits 28-31.
    The ASIC ADC is inverted, so HG and LG are flipped with 0x3FFF - field.
    """

    def __init__(self, bin_file, output_dir=None):

        self.file_path = os.path.abspath(bin_file)

        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"BIN file not found:\n{self.file_path}")

        stem = os.path.splitext(os.path.basename(self.file_path))[0]
        self.run_name = stem[5:] if stem.startswith("data_") else stem

        # Results go to <bin folder>/<run name>/ unless told otherwise
        if output_dir is None:
            output_dir = os.path.join(os.path.dirname(self.file_path),
                                      self.run_name)
        self.output_dir = os.path.abspath(output_dir)

        self.plot_dir = os.path.join(self.output_dir, "plots")
        os.makedirs(self.plot_dir, exist_ok=True)

        self.csv_file = os.path.join(self.output_dir,
                                     f"channel_summary_{self.run_name}.csv")

        self.datalen = os.path.getsize(self.file_path)

        # ASICs to drop entirely. Their bytes are still consumed while
        # parsing (the stream has to stay aligned) but nothing is stored,
        # so these channels are absent from the summary CSV, the camera
        # maps and the finger plots alike.
        self.excluded = set(BAD_ASICS) if EXCLUDE_BAD_ASIC else set()

        # One flat array of words per ASIC, 32 per event in channel order.
        # Excluded ASICs keep an empty array so nothing downstream KeyErrors.
        self.channels = {p: array("I") for p in PREFIXES}

        self.events_per_daq = {d: 0 for d in DAQS}
        self.first_timestamp = None
        self.last_timestamp = None
        self.n_packs = 0

    # --------------------------------------------------------
    # Per-channel access
    # --------------------------------------------------------

    def channel_words(self, daq, asic, ch):
        """Raw words for one channel, one per event."""
        words = self.channels[f"D{daq}_{asic}"]
        if len(words) == 0:
            return np.array([], dtype=np.int64)
        return np.frombuffer(words, dtype=np.uint32)[ch::32].astype(np.int64)

    def channel_values(self, daq, asic, ch, gain="HG"):
        """Decoded ADC values for one channel: gain is 'HG', 'LG' or 'HIT'."""
        w = self.channel_words(daq, asic, ch)
        if w.size == 0:
            return w

        gain = gain.upper()
        if gain == "HG":
            return (0x3FFF - w) & 0x3FFF
        if gain == "LG":
            return (0x3FFF - (w >> 14)) & 0x3FFF
        if gain == "HIT":
            return (w >> 28) & 0xF
        raise ValueError(f"gain must be HG, LG or HIT, got {gain!r}")

    def histogram(self, daq, asic, ch, gain="HG"):
        """ADC histogram for one channel, rebinned by REBIN_FACTOR."""
        vals = self.channel_values(daq, asic, ch, gain)
        n_out = NBINS // REBIN_FACTOR

        if vals.size == 0:
            return (np.zeros(n_out),
                    np.arange(n_out) * REBIN_FACTOR + REBIN_FACTOR / 2)

        hist, edges = np.histogram(vals, bins=NBINS, range=(0, NBINS))

        if REBIN_FACTOR > 1:
            hist = hist[:n_out * REBIN_FACTOR].reshape(n_out,
                                                       REBIN_FACTOR).sum(axis=1)
            centers = edges[:n_out * REBIN_FACTOR].reshape(n_out,
                                                           REBIN_FACTOR).mean(axis=1)
        else:
            centers = (edges[:-1] + edges[1:]) / 2

        return hist, centers

    # --------------------------------------------------------
    # Payload parsing
    # --------------------------------------------------------

    def parse_terzina_payload(self, payload):
        """Walk one payload's DAQ blocks, appending channel words as it goes."""
        offset = 0

        def u32():
            nonlocal offset
            if offset + 4 > len(payload):
                raise ValueError("Unexpected end of payload while reading u32")
            val = int.from_bytes(payload[offset:offset + 4], "little")
            offset += 4
            return val

        def u64():
            nonlocal offset
            if offset + 8 > len(payload):
                raise ValueError("Unexpected end of payload while reading u64")
            val = int.from_bytes(payload[offset:offset + 8], "little")
            offset += 8
            return val

        def skip(n):
            """Advance past n bytes without decoding them."""
            nonlocal offset
            if offset + n > len(payload):
                raise ValueError("Unexpected end of payload while skipping")
            offset += n

        timestamp = u64()
        if self.first_timestamp is None:
            self.first_timestamp = timestamp
        self.last_timestamp = timestamp

        if u64() != CONC_HEADER:            # ID_CONC
            return

        while offset < len(payload):

            daq_id = u32()

            if daq_id in DAQ_IDS:

                d = DAQ_IDS[daq_id]

                u32(), u32(), u32()         # trigger counter, data1, data2

                for a in ASICS:
                    if (d, a) in self.excluded:
                        skip(32 * 4)        # consume, but store nothing
                    else:
                        self.channels[f"D{d}_{a}"].extend(
                            u32() for _ in range(32)
                        )

                u64(), u64()                # LOST, REAL

                self.events_per_daq[d] += 1

            elif daq_id == 0xDEADBEEF:
                break

            else:
                print(f"UNKNOWN DAQ ID : {hex(daq_id)}")
                break

    # --------------------------------------------------------
    # File parsing
    # --------------------------------------------------------

    def process_pck(self, max_packets=None):

        print()
        print("===================================")
        print("PROCESSING BINARY FILE")
        print("===================================")
        print(f"INPUT : {self.file_path}")
        print(f"SIZE  : {self.datalen:,} bytes")

        if self.excluded:
            listed = ", ".join(f"DAQ{d} ASIC-{a}"
                               for d, a in sorted(self.excluded))
            print(f"DROPPED (EXCLUDE_BAD_ASIC = True): {listed}")
            print("        -> absent from the CSV, camera maps and finger plots")
        print()

        start_time = time.perf_counter()

        with open(self.file_path, "rb") as file:

            while True:

                header = file.read(12)
                if len(header) < 12:
                    break

                ID_pack = int.from_bytes(header[0:4], "little")
                len_pack = int.from_bytes(header[4:8], "little")
                pack_counter = int.from_bytes(header[8:12], "little")

                if ID_pack != 0xA77A:
                    print(f"INVALID PACK ID : {hex(ID_pack)}")
                    continue

                payload = file.read(len_pack)
                if len(payload) != len_pack:
                    print("ERROR: TRUNCATED PACK")
                    break

                try:
                    self.parse_terzina_payload(payload)
                    self.n_packs += 1
                except Exception as e:
                    print(f"ERROR PACK {pack_counter}: {e}")

                if self.n_packs % 10000 == 0:
                    print(".", end="", flush=True)

                if max_packets and self.n_packs >= max_packets:
                    break

        elapsed = time.perf_counter() - start_time

        print()
        print("PROCESSING FINISHED")
        print(f"PACKETS : {self.n_packs}")
        for d in DAQS:
            print(f"  DAQ{d}: {self.events_per_daq[d]} events")
        print(f"TIME    : {elapsed:.3f} s")

        return elapsed

    # --------------------------------------------------------
    # Channel summary CSV
    # --------------------------------------------------------

    def export_channel_summary(self):
        """
        One row per channel: HIT count, HG mean, LG mean.

        Means are over values > 0, matching the original script; the whole
        thing is vectorised, so it runs in well under a second instead of
        tens of seconds.
        """
        print()
        print("===================================")
        print("CREATING CHANNEL SUMMARY")
        print("===================================")

        start_time = time.perf_counter()

        with open(self.csv_file, "w", newline="") as file:

            writer = csv.writer(file, delimiter=";")
            writer.writerow(["DAQ_CODE", "HIT", "HG_MEAN", "LG_MEAN"])

            for prefix in PREFIXES:

                daq = int(prefix[1])
                asic = prefix[3]

                if len(self.channels[prefix]) == 0:
                    continue

                for ch in range(32):

                    w = self.channel_words(daq, asic, ch)
                    if w.size == 0:
                        continue

                    hg = (0x3FFF - w) & 0x3FFF
                    lg = (0x3FFF - (w >> 14)) & 0x3FFF
                    hit = (w >> 28) & 0xF

                    hg_valid = hg[hg > 0]
                    lg_valid = lg[lg > 0]

                    hg_mean = float(hg_valid.mean()) if hg_valid.size else 0.0
                    lg_mean = float(lg_valid.mean()) if lg_valid.size else 0.0
                    hit_count = int((hit > 0).sum())

                    writer.writerow([
                        f"{prefix}_HG_{ch:02d}",
                        hit_count,
                        round(hg_mean, 2),
                        round(lg_mean, 2),
                    ])

        elapsed = time.perf_counter() - start_time
        print(f"SAVED : {self.csv_file}")
        print(f"SUMMARY TIME : {elapsed:.3f} s")

        return elapsed


# ============================================================
# PIN-TO-PIN MAPPING
# ============================================================

def filter_from_daq(pinmap, daq_signal):

    parts = daq_signal.strip().split("_")

    D_part = parts[0].strip()
    ASIC = parts[1].strip()
    IN = str(int(parts[3].strip()))

    quadrant = int(D_part[1:]) - 1
    in_signal = f"In{ASIC}{IN}"

    df = pinmap.copy()
    df["daq_signal"] = df["daq_signal"].astype(str).str.strip()
    df["QUADRANT"] = pd.to_numeric(df["QUADRANT"], errors="coerce")

    return df[(df["QUADRANT"] == quadrant) & (df["daq_signal"] == in_signal)]


def load_quadrant_csv(path):

    df_raw = pd.read_csv(path, header=None, dtype=str, encoding="latin1")

    header_idx = df_raw.apply(
        lambda r: r.astype(str).str.contains("QUADRANT", case=False, na=False)
    ).any(axis=1).idxmax()

    df = df_raw.iloc[header_idx + 1:].reset_index(drop=True)

    rows = []
    block_size = 10

    for _, row in df.iterrows():
        row = row.fillna("")
        for i in range(0, len(row), block_size + 1):
            block = row[i:i + block_size]
            if len(block) < block_size:
                continue
            if block.iloc[0] == "":
                continue
            rows.append(block.tolist())

    columns = [
        "QUADRANT", "TILE", "10TA.J#", "pin of 10TA.J",
        "Hierachical signal name", "signal name",
        "CB.J", "pin of CB.J", "daq_signal", "GLOBAL ID",
    ]

    return pd.DataFrame(rows, columns=columns)


def read_config(filename):

    config = {}

    with open(filename, "r") as f:
        for line in f:
            line = line.split("#")[0].strip()
            if not line or ":" not in line:
                continue
            k, v = line.split(":", 1)
            k, v = k.strip(), v.strip()
            try:
                v = float(v) if "." in v else int(v)
            except ValueError:
                pass
            config[k] = v

    return config


def get_asic(daq, tile, chid):

    side = 0 if chid < 32 else 1
    t = tile % 5
    pattern = [0, 2, 4, 2, 0] if side == 0 else [1, 3, 4, 3, 1]
    return pattern[t]


# ============================================================
# GEOMETRY
# ============================================================

def tile_ch_to_pos_and_global(tile_ID, ch_local_ID, config):

    nx_tiles = config["nx_tiles"]
    ny_tiles = config["ny_tiles"]
    Nx_tot = config["nx_sipm_pixel"]
    Ny_tot = config["ny_sipm_pixel"]

    tile_row_id = ny_tiles - tile_ID // nx_tiles - 1

    if tile_row_id % 2 == 0:
        tile_col_id = nx_tiles - tile_ID % nx_tiles - 1
    else:
        tile_col_id = tile_ID % nx_tiles

    ch_local_ID = int(ch_local_ID)

    if tile_row_id % 2 == 0:
        ch_local_ID = Nx_tot * Ny_tot - 1 - ch_local_ID

    dTx = (Nx_tot * config["sensitive_sipm_pixel_sizeX"]
           + (Nx_tot - 1) * config["sipm_pixel_pitch_right"]
           + 2 * config["sipm_array_d"])

    dTy = (Ny_tot * config["sensitive_sipm_pixel_sizeY"]
           + int(Ny_tot / 2) * config["sipm_pixel_pitch_down"]
           + int((Ny_tot - 1) / 2) * config["sipm_pixel_pitch_up"]
           + 2 * config["sipm_array_d"])

    tile_posX = (-nx_tiles * dTx / 2
                 - (nx_tiles - 1) * config["sipm_array_d"] / 2
                 + dTx / 2
                 + (dTx + config["sipm_array_d"]) * tile_col_id)

    tile_posY = (-ny_tiles * dTy / 2
                 - (ny_tiles - 1) * config["sipm_array_d"] / 2
                 + dTy / 2
                 + (dTy + config["sipm_array_d"]) * tile_row_id)

    pixel_local_col = ch_local_ID // Nx_tot
    pixel_local_row = Ny_tot - 1 - (ch_local_ID % Nx_tot)

    pixel_posX = (tile_posX
                  - Nx_tot * config["sensitive_sipm_pixel_sizeX"] / 2
                  - (Nx_tot - 1) * config["sipm_pixel_pitch_right"] / 2
                  + config["sensitive_sipm_pixel_sizeX"] / 2
                  + (config["sensitive_sipm_pixel_sizeX"]
                     + config["sipm_pixel_pitch_right"]) * pixel_local_col)

    pixel_posY = (tile_posY
                  - Ny_tot * config["sensitive_sipm_pixel_sizeY"] / 2
                  - int(Ny_tot / 2) * config["sipm_pixel_pitch_down"] / 2
                  - int((Ny_tot - 1) / 2) * config["sipm_pixel_pitch_up"] / 2
                  + (2 * pixel_local_row + 1)
                  * config["sensitive_sipm_pixel_sizeY"] / 2
                  + config["sipm_pixel_pitch_down"]
                  * int((pixel_local_row + 1) / 2)
                  + config["sipm_pixel_pitch_up"]
                  * int(pixel_local_row / 2))

    ncols_total = nx_tiles * Nx_tot
    global_col = tile_col_id * Nx_tot + pixel_local_col
    global_row = tile_row_id * Ny_tot + pixel_local_row
    globalID = global_row * ncols_total + global_col

    return (round(pixel_posX, 2), round(pixel_posY, 2), int(globalID),
            round(tile_posX, 2), round(tile_posY, 2))


# ============================================================
# BUILD COMPLETE MAPPING
# ============================================================

def build_complete_mapping(mapping_dir, out_dir=None):
    """Load the four QUADRANT CSVs and the .conf, and build the pixel map."""

    print()
    print("===================================")
    print("LOADING PIN-TO-PIN MAPPING")
    print("===================================")

    mapping_dir = os.path.abspath(mapping_dir)

    if not os.path.isdir(mapping_dir):
        raise FileNotFoundError(f"Mapping directory not found:\n{mapping_dir}")

    files = [os.path.join(mapping_dir, n) for n in MAPPING_FILES]
    conf_path = os.path.join(mapping_dir, CONF_FILE)

    missing = [p for p in files + [conf_path] if not os.path.isfile(p)]
    if missing:
        raise FileNotFoundError(
            f"Missing from {mapping_dir}:\n  "
            + "\n  ".join(os.path.basename(m) for m in missing)
        )

    print(f"DIR : {mapping_dir}")

    pinmap = pd.concat([load_quadrant_csv(f) for f in files],
                       ignore_index=True)
    config = read_config(conf_path)

    # --- signal parsing ---
    col = pinmap["signal name"].astype("string").str.strip()
    parsed = col.str.extract(r"^([A-F])(\d+)_CH(\d+)$")
    valid_mask = parsed[0].notna()

    pin_valid = pinmap.loc[valid_mask].copy()
    parsed_valid = parsed.loc[valid_mask].copy()

    complete_data = pd.DataFrame()
    complete_data["QUADRANT"] = pin_valid["QUADRANT"].reset_index(drop=True)
    complete_data["TILE #"] = pd.to_numeric(
        pin_valid["TILE"], errors="coerce").reset_index(drop=True)
    complete_data["CHANNEL #"] = pd.to_numeric(
        parsed_valid[2], errors="coerce").reset_index(drop=True)
    complete_data["signal name"] = pin_valid["signal name"].reset_index(drop=True)
    complete_data["GLOBAL ID"] = pin_valid["GLOBAL ID"].reset_index(drop=True)

    complete_data["ASIC #"] = complete_data.apply(
        lambda row: get_asic(int(row["QUADRANT"]), int(row["TILE #"]),
                             int(row["CHANNEL #"])),
        axis=1,
    )

    results = complete_data.apply(
        lambda r: tile_ch_to_pos_and_global(int(r["TILE #"]),
                                            int(r["CHANNEL #"]), config),
        axis=1, result_type="expand",
    )
    results.columns = ["PixelCentrePosX_mm", "PixelCentrePosY_mm",
                       "GlobalID", "TileCentrePosX_mm", "TileCentrePosY_mm"]

    complete_data = pd.concat([complete_data, results], axis=1)

    mapping_file = f"complete_mapping_{config['nx_tiles']}x{config['ny_tiles']}.csv"
    if out_dir is not None:
        mapping_file = os.path.join(out_dir, mapping_file)

    complete_data.to_csv(mapping_file, index=False, float_format="%.2f")
    print(f"MAPPING SAVED : {mapping_file}")

    return pinmap, complete_data, config


# ============================================================
# CAMERA MAP
# ============================================================

def _pixel_geometry(pinmap, config, daq_codes):
    """
    Focal-plane geometry for a list of DAQ_CODEs.

    Returns a DataFrame with PixelCentrePosX_mm / PixelCentrePosY_mm, or
    None if none of the codes map to a pixel.
    """
    rows = []

    for daq_code in daq_codes:

        data_NI = filter_from_daq(pinmap, daq_code)
        if len(data_NI) == 0:
            continue

        col = data_NI["signal name"].astype("string").str.strip()
        parsed = col.str.extract(r"^([A-F])(\d+)_CH(\d+)$")
        valid_mask = parsed[0].notna()

        ni_valid = data_NI.loc[valid_mask].copy()
        parsed_valid = parsed.loc[valid_mask].copy()

        tmp = pd.DataFrame()
        tmp["TILE #"] = pd.to_numeric(ni_valid["TILE"], errors="coerce").values
        tmp["CHANNEL #"] = pd.to_numeric(parsed_valid[2], errors="coerce").values

        if not tmp.empty:
            rows.append(tmp)

    if not rows:
        return None

    out = pd.concat(rows, ignore_index=True)

    res = out.apply(
        lambda r: tile_ch_to_pos_and_global(int(r["TILE #"]),
                                            int(r["CHANNEL #"]), config),
        axis=1, result_type="expand",
    )
    res.columns = ["PixelCentrePosX_mm", "PixelCentrePosY_mm",
                   "GlobalID", "TileCentrePosX_mm", "TileCentrePosY_mm"]

    return pd.concat([out, res], axis=1)


def plot_camera_map(pinmap, complete_data, config, csv_file,
                    value_column, title, output_file, excluded=None):

    print()
    print(f"CREATING MAP : {value_column}")

    summary = pd.read_csv(csv_file, sep=";")
    rows = []

    for _, row in summary.iterrows():

        daq_code = row["DAQ_CODE"]
        value = row[value_column]

        if pd.isna(value):
            continue
        value = float(value)
        if value < 0:
            continue

        data_NI = filter_from_daq(pinmap, daq_code)
        if len(data_NI) == 0:
            continue

        col = data_NI["signal name"].astype("string").str.strip()
        parsed = col.str.extract(r"^([A-F])(\d+)_CH(\d+)$")
        valid_mask = parsed[0].notna()

        ni_valid = data_NI.loc[valid_mask].copy()
        parsed_valid = parsed.loc[valid_mask].copy()

        tmp = pd.DataFrame()
        tmp["QUADRANT"] = ni_valid["QUADRANT"].astype(int).values
        tmp["TILE #"] = pd.to_numeric(ni_valid["TILE"], errors="coerce").values
        tmp["CHANNEL #"] = pd.to_numeric(parsed_valid[2], errors="coerce").values
        tmp["VALUE"] = value

        if not tmp.empty:
            rows.append(tmp)

    if not rows:
        print(f"No valid data for {value_column}")
        return None

    mini = pd.concat(rows, ignore_index=True)

    res = mini.apply(
        lambda r: tile_ch_to_pos_and_global(int(r["TILE #"]),
                                            int(r["CHANNEL #"]), config),
        axis=1, result_type="expand",
    )
    res.columns = ["PixelCentrePosX_mm", "PixelCentrePosY_mm",
                   "GlobalID", "TileCentrePosX_mm", "TileCentrePosY_mm"]
    mini = pd.concat([mini, res], axis=1)

    # ---- PLOT ----
    fig, ax = plt.subplots(figsize=(10, 5), dpi=150)

    min_value = mini["VALUE"].min()
    max_value = mini["VALUE"].max()

    # Avoid a degenerate colour scale when every value is the same
    if min_value == max_value:
        if min_value == 0:
            max_value = 1
        else:
            min_value = 0

    norm = plt.Normalize(vmin=min_value, vmax=max_value)
    cmap = plt.cm.jet

    sx = config["sensitive_sipm_pixel_sizeX"]
    sy = config["sensitive_sipm_pixel_sizeY"]

    # Empty pixels
    for _, row in complete_data.iterrows():
        ax.add_patch(patches.Rectangle(
            (row["PixelCentrePosX_mm"] - sx / 2,
             row["PixelCentrePosY_mm"] - sy / 2),
            sx, sy,
            facecolor="whitesmoke", edgecolor="lightgray", linewidth=0.25,
        ))

    # Excluded ASICs: their channels were never recorded, so they would
    # otherwise show as a grey hole. Draw them at the BOTTOM of the colour
    # scale instead, so they blend in with the unhit pixels. They still
    # contribute nothing to the data, the scale or the analysis.
    if excluded:
        codes = [f"D{d}_{a}_HG_{c:02d}"
                 for d, a in sorted(excluded) for c in range(32)]
        exc = _pixel_geometry(pinmap, config, codes)

        if exc is not None:
            floor = cmap(norm(min_value))
            for _, row in exc.iterrows():
                ax.add_patch(patches.Rectangle(
                    (row["PixelCentrePosX_mm"] - sx / 2,
                     row["PixelCentrePosY_mm"] - sy / 2),
                    sx, sy,
                    facecolor=floor, edgecolor="black", linewidth=0.4,
                ))

    # Active pixels
    for _, row in mini.iterrows():
        ax.add_patch(patches.Rectangle(
            (row["PixelCentrePosX_mm"] - sx / 2,
             row["PixelCentrePosY_mm"] - sy / 2),
            sx, sy,
            facecolor=cmap(norm(row["VALUE"])),
            edgecolor="black", linewidth=0.4,
        ))

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label(value_column, fontsize=14)

    ax.set_aspect("equal")

    xlim = np.max(complete_data["PixelCentrePosX_mm"]) + 5
    ylim = np.max(complete_data["PixelCentrePosY_mm"]) + 5
    ax.set_xlim(-xlim, xlim)
    ax.set_ylim(-ylim, ylim)

    ax.set_xlabel("x (mm)")
    ax.set_ylabel("y (mm)")
    ax.set_title(title)
    ax.grid(True, linestyle="--", linewidth=0.2)

    fig.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"SAVED : {output_file}")
    return output_file


# ============================================================
# BRIGHTEST CHANNELS — most resolved p.e. peaks
# ============================================================

def gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu) ** 2 / (2 * sigma ** 2))


def fit_peak(x, y, idx, window=GAUSS_FIT_WINDOW):
    """Gaussian-fit one peak; fall back to the bin position if the fit fails."""
    lo = max(0, idx - window)
    hi = min(len(x), idx + window + 1)
    xw, yw = x[lo:hi], y[lo:hi]
    try:
        popt, _ = curve_fit(gaussian, xw, yw,
                            p0=[yw.max(), x[idx], (xw[-1] - xw[0]) / 4],
                            maxfev=5000)
        return float(popt[1])
    except Exception:
        return float(x[idx])


def analyse_channel(hist, bins):
    """Return (peak indices, fitted positions, gain) for one spectrum."""
    ys = gaussian_filter1d(hist.astype(float), sigma=SMOOTH_SIGMA)

    peaks, _ = find_peaks(ys,
                          prominence=PEAK_PROMINENCE * max(ys.max(), 1),
                          distance=PEAK_DISTANCE)
    if len(peaks) == 0:
        return peaks, [], None

    positions = [fit_peak(bins, ys, p) for p in peaks]

    g = None
    sp = np.diff(positions)
    if len(sp) > 0:
        med = np.median(sp)
        good = (sp > 0.5 * med) & (sp < 2.0 * med)
        if good.sum() > 0:
            g = float(np.mean(sp[good]))

    return peaks, positions, g


def rank_channels(data_proc, gain="HG", min_peaks=BRIGHT_MIN_PEAKS):
    """
    Rank every channel by how much of the p.e. ladder it resolves.

    Key: (number of peaks, ADC span of the ladder), best first. The span
    breaks ties, favouring the channel whose peaks reach higher.
    """
    gain = gain.upper()

    # Excluded ASICs were never stored, so they cannot appear here anyway.
    # The explicit check just keeps the reported pool size honest.
    skip = data_proc.excluded

    print()
    print("===================================")
    print(f"SCANNING FOR BRIGHTEST CHANNELS ({gain})")
    print("===================================")
    if skip:
        listed = ", ".join(f"DAQ{d} ASIC-{a}" for d, a in sorted(skip))
        print(f"EXCLUDED: {listed}")

    start_time = time.perf_counter()

    out = []
    n_skipped = 0

    for daq in DAQS:
        if data_proc.events_per_daq[daq] == 0:
            continue
        for asic in ASICS:

            if (daq, asic) in skip:
                n_skipped += 32
                continue

            for ch in range(32):

                hist, bins = data_proc.histogram(daq, asic, ch, gain)
                if hist.max() == 0:
                    continue

                peaks, positions, g = analyse_channel(hist, bins)
                if len(peaks) < min_peaks:
                    continue

                nz = np.where(hist > 0)[0]
                out.append({
                    "DAQ": daq, "ASIC": asic, "CH": ch,
                    "N_PEAKS": int(len(peaks)),
                    "LADDER": float(positions[-1] - positions[0]),
                    "GAIN": g,
                    "ADC_MIN": float(bins[nz.min()]),
                    "ADC_MAX": float(bins[nz.max()]),
                })

    out.sort(key=lambda r: (r["N_PEAKS"], r["LADDER"]), reverse=True)

    pool = 640 - n_skipped
    print(f"Channels with at least {min_peaks} peaks: {len(out)} / {pool}"
          f"{f' (skipped {n_skipped})' if n_skipped else ''}")
    print(f"SCAN TIME : {time.perf_counter() - start_time:.3f} s")

    return out


def plot_single_finger(data_proc, daq, asic, ch, gain, save_path):
    """Full-size finger plot for one channel, peaks labelled by p.e. number."""
    hist, bins = data_proc.histogram(daq, asic, ch, gain)
    peaks, _, g = analyse_channel(hist, bins)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(bins, hist, "b-", lw=0.8, alpha=0.7, label="Raw")

    if len(peaks) > 0:
        ax.plot(bins[peaks], hist[peaks], "go", markersize=8,
                label="Detected peaks")
        for i, p in enumerate(peaks):
            ax.annotate(f"{i + 1} p.e.\n({bins[p]:.0f})",
                        xy=(bins[p], hist[p]), xytext=(0, 12),
                        textcoords="offset points", ha="center",
                        fontsize=8, color="darkgreen", fontweight="bold")

    nz = np.where(hist > 0)[0]
    if len(nz) > 0:
        ax.set_xlim(bins[nz.min()] - 50, bins[nz.max()] + 50)

    ax.set_yscale("log")
    ax.set_xlabel("ADC bin", fontsize=13)
    ax.set_ylabel("Counts", fontsize=13)
    gain_txt = f", gain {g:.1f} ADC/p.e." if g is not None else ""
    ax.set_title(f"DAQ{daq} ASIC-{asic} CH{ch} {gain}"
                 f"  ({len(peaks)} peaks{gain_txt})", fontsize=14)
    ax.legend(fontsize=11, loc="upper right")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"SAVED : {save_path}")


def plot_bright_grid(data_proc, ranked, gain, n_top=BRIGHT_N_TOP,
                     ncols=BRIGHT_NCOLS, save_path=None):
    """Grid of the best channels' spectra."""

    sel = ranked[:n_top]
    if not sel:
        print("No channel resolved enough peaks — nothing to plot.")
        return None, []

    print()
    print(f"  {'RANK':>4}  {'CHANNEL':<18} {'PEAKS':>5} {'LADDER':>8} "
          f"{'GAIN':>7}  {'ADC RANGE':>16}")
    print("  " + "-" * 64)

    nrows = int(np.ceil(len(sel) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5.0 * ncols, 2.9 * nrows),
                             squeeze=False)

    for i, r in enumerate(sel):

        ax = axes[i // ncols][i % ncols]
        d, a, c = r["DAQ"], r["ASIC"], r["CH"]

        hist, bins = data_proc.histogram(d, a, c, gain)
        peaks, _, _ = analyse_channel(hist, bins)

        ax.plot(bins, hist, "-", color="darkblue", lw=0.7)
        if len(peaks) > 0:
            ax.plot(bins[peaks], hist[peaks], "o", color="darkgreen",
                    markersize=4)

        nz = np.where(hist > 0)[0]
        if len(nz) > 0:
            pad = max(10, 0.02 * (bins[nz.max()] - bins[nz.min()]))
            ax.set_xlim(bins[nz.min()] - pad, bins[nz.max()] + pad)

        ax.set_yscale("log")
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.25)

        gain_txt = f"{r['GAIN']:.1f}" if r["GAIN"] is not None else "n/a"
        ax.set_title(f"DAQ{d} ASIC-{a} CH{c}   ({r['N_PEAKS']} pk, "
                     f"gain {gain_txt})", fontsize=9)

        print(f"  {i + 1:>4}  DAQ{d} ASIC-{a} CH{c:<6} {r['N_PEAKS']:>5} "
              f"{r['LADDER']:>8.1f} {gain_txt:>7}  "
              f"{r['ADC_MIN']:>7.0f}-{r['ADC_MAX']:<8.0f}")

    for j in range(len(sel), nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")

    fig.suptitle(f"TERZINA FPA — Top {len(sel)} Channels by resolved p.e. peaks "
                 f"({gain})", fontsize=14, y=0.995)
    fig.supxlabel("ADC bin", fontsize=12)
    fig.supylabel("Counts", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.975])

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"\nSAVED : {save_path}")

    plt.close(fig)
    return fig, sel


# ============================================================
# MAIN
# ============================================================

def main(bin_file=BIN_FILE, mapping_dir=MAPPING_DIR,
         gain=BRIGHT_GAIN, n_top=BRIGHT_N_TOP):

    total_start = time.perf_counter()

    gain = gain.upper()
    if gain not in ("HG", "LG"):
        raise ValueError(f"gain must be 'HG' or 'LG', got {gain!r}")

    # --- parse once -----------------------------------------
    data_proc = DataProcessing(bin_file)

    print(f"RUN        : {data_proc.run_name}")
    print(f"OUTPUT DIR : {data_proc.output_dir}")

    data_proc.process_pck(MAX_PACKETS)
    data_proc.export_channel_summary()

    # --- mapping --------------------------------------------
    pinmap, complete_data, config = build_complete_mapping(
        mapping_dir, out_dir=data_proc.output_dir)

    # --- camera maps ----------------------------------------
    print()
    print("===================================")
    print("CREATING CAMERA MAPS")
    print("===================================")

    maps = [
        ("HIT", "Camera HIT Occupancy", "Camera_HIT_Occupancy.png"),
        ("HG_MEAN", "Camera HG Mean", "Camera_HG_Mean.png"),
        ("LG_MEAN", "Camera LG Mean", "Camera_LG_Mean.png"),
    ]

    for value_column, title, filename in maps:
        try:
            plot_camera_map(pinmap, complete_data, config,
                            data_proc.csv_file, value_column, title,
                            os.path.join(data_proc.plot_dir, filename),
                            excluded=data_proc.excluded)
        except Exception as e:
            print(f"FAILED : {value_column} ({type(e).__name__}: {e})")

    # --- brightest-channel finger plots ----------------------
    ranked = rank_channels(data_proc, gain)

    sel = []
    if ranked:
        _, sel = plot_bright_grid(
            data_proc, ranked, gain, n_top=n_top,
            save_path=os.path.join(data_proc.plot_dir,
                                   f"bright_fingerplots_{gain}.png"))

        for r in sel[:BRIGHT_N_INDIVIDUAL]:
            d, a, c = r["DAQ"], r["ASIC"], r["CH"]
            plot_single_finger(
                data_proc, d, a, c, gain,
                os.path.join(data_proc.plot_dir,
                             f"bright_DAQ{d}_ASIC{a}_CH{c}_{gain}.png"))

    # --- done ------------------------------------------------
    print()
    print("===================================")
    print("DONE")
    print("===================================")
    print(f"TOTAL TIME      : {time.perf_counter() - total_start:.3f} s")
    print()
    print(f"CHANNEL SUMMARY : {data_proc.csv_file}")
    print(f"PLOTS DIRECTORY : {data_proc.plot_dir}")
    print()

    return data_proc, ranked


if __name__ == "__main__":

    # Jupyter and Spyder also report __name__ == "__main__", and they fill
    # sys.argv with their own kernel options, so only trust argv when this is
    # really running as a command-line script.
    args = [] if "ipykernel" in sys.modules else sys.argv[1:]

    path = args[0] if len(args) > 0 else BIN_FILE
    mdir = args[1] if len(args) > 1 else MAPPING_DIR

    main(path, mdir)

RUN        : 07_03-11_28
OUTPUT DIR : C:\Users\muhda\Desktop\Turin_Mission\Data\07_03-11_28

PROCESSING BINARY FILE
INPUT : C:\Users\muhda\Desktop\Turin_Mission\Data\data_07_03-11_28.bin
SIZE  : 339,054,448 bytes
DROPPED (EXCLUDE_BAD_ASIC = True): DAQ3 ASIC-D
        -> absent from the CSV, camera maps and finger plots

............
PROCESSING FINISHED
PACKETS : 124856
  DAQ1: 124841 events
  DAQ2: 124835 events
  DAQ3: 124825 events
  DAQ4: 124815 events
TIME    : 73.814 s

CREATING CHANNEL SUMMARY
SAVED : C:\Users\muhda\Desktop\Turin_Mission\Data\07_03-11_28\channel_summary_07_03-11_28.csv
SUMMARY TIME : 3.096 s

LOADING PIN-TO-PIN MAPPING
DIR : C:\Users\muhda\Desktop\Software\GitHub\Terzina_data_ana\mapping_ana
MAPPING SAVED : C:\Users\muhda\Desktop\Turin_Mission\Data\07_03-11_28\complete_mapping_5x2.csv

CREATING CAMERA MAPS

CREATING MAP : HIT
SAVED : C:\Users\muhda\Desktop\Turin_Mission\Data\07_03-11_28\plots\Camera_HIT_Occupancy.png

CREATING MAP : HG_MEAN
SAVED : C:\Users\muhda